# Music Genre Classification — Deep Learning with ResNet18

Classifies music into 10 genres (blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock) using mel-spectrogram images fed through a pre-trained ResNet18 network.

**Pipeline:** Raw audio → 5-second crop → Mel spectrogram → ResNet18 (fine-tuned) → genre label

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/genres_original'

## 2. Configuration

In [ ]:
drive.mount('/content/drive')

# ==========================================
# CONFIGURATION - UPDATE THIS PATH!
# ==========================================

Mounted at /content/drive


class AudioSpectrogramDataset(Dataset):
    def __init__(self, file_paths, is_training=True):
        self.file_paths = file_paths
        self.is_training = is_training

        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=22050,
            n_fft=2048,
            hop_length=512,
            n_mels=128
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        self.time_masking = T.TimeMasking(time_mask_param=30)
        self.freq_masking = T.FrequencyMasking(freq_mask_param=15)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label_str = os.path.basename(os.path.dirname(file_path))
        label = CLASS_TO_IDX[label_str]

        waveform, sr = torchaudio.load(file_path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != 22050:
            resampler = T.Resample(sr, 22050)
            waveform = resampler(waveform)

        waveform = waveform / (waveform.abs().max() + 1e-9)

        total_samples = waveform.shape[1]

        if total_samples > SAMPLES_PER_TRACK:
            if self.is_training:
                max_start = total_samples - SAMPLES_PER_TRACK
                start = random.randint(0, max_start)
            else:
                start = (total_samples - SAMPLES_PER_TRACK) // 2
            waveform = waveform[:, start:start + SAMPLES_PER_TRACK]
        else:
            pad_amount = SAMPLES_PER_TRACK - total_samples
            waveform = torch.nn.functional.pad(waveform, (0, pad_amount))

        mel_spec = self.mel_spectrogram(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)
        mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-6)

        return mel_spec_db, label

In [ ]:
DATA_DIR = '/content/drive/MyDrive/genres_original'

## 4. Model Architecture

### Approach 1: Custom CNN (Baseline)

A lightweight 4-block CNN built from scratch. Explored as a baseline before moving to transfer learning.

print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("Training complete. Saving model...")
torch.save(model.state_dict(), '/content/drive/MyDrive/music_cnn_res_model.pth')
print("Model saved.")

## 6. Inference

### Approach 2: Pre-trained ResNet18 (Transfer Learning — Final Model)

ResNet18 pre-trained on ImageNet, adapted for single-channel mel spectrograms. The first conv layer is modified to accept 1 channel instead of 3, and the final classification layer is replaced with a 10-class output.

In [ ]:
# ==========================================
# 2. DATASET DEFINITION (Upgraded with Augmentations)
# ==========================================
class AudioSpectrogramDataset(Dataset):
    def __init__(self, file_paths, is_training=True):
        self.file_paths = file_paths
        self.is_training = is_training

        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=22050,
            n_fft=2048,
            hop_length=512,
            n_mels=128
        )

        self.amplitude_to_db = T.AmplitudeToDB()

        self.time_masking = T.TimeMasking(time_mask_param=30)
        self.freq_masking = T.FrequencyMasking(freq_mask_param=15)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]

        label_str = os.path.basename(os.path.dirname(file_path))
        label = CLASS_TO_IDX[label_str]

        waveform, sr = torchaudio.load(file_path)

        # Convert stereo -> mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # IMPORTANT: RESAMPLE
        if sr != 22050:
            resampler = T.Resample(sr, 22050)
            waveform = resampler(waveform)
            sr = 22050

        # Normalize volume
        waveform = waveform / (waveform.abs().max() + 1e-9)

        total_samples = waveform.shape[1]

        if total_samples > SAMPLES_PER_TRACK:
            if self.is_training:
                max_start = total_samples - SAMPLES_PER_TRACK
                start = random.randint(0, max_start)
            else:
                start = (total_samples - SAMPLES_PER_TRACK) // 2

            waveform = waveform[:, start:start + SAMPLES_PER_TRACK]

        else:
            pad_amount = SAMPLES_PER_TRACK - total_samples
            waveform = torch.nn.functional.pad(waveform, (0, pad_amount))

        mel_spec = self.mel_spectrogram(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)

        print("TRAIN SHAPE:", mel_spec_db.shape)
        print("TRAIN MIN/MAX:", mel_spec_db.min().item(), mel_spec_db.max().item())
        print("TRAIN MEAN/STD:", mel_spec_db.mean().item(), mel_spec_db.std().item())
        mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-6)

        return mel_spec_db, label

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16) # NEW: Batch Normalization
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)

        # Conv Block 2
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32) # NEW
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)

        # Conv Block 3
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64) # NEW
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2)

        # Conv Block 4
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128) # NEW
        self.relu4 = nn.ReLU()

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128, 64)
        self.bn_fc1 = nn.BatchNorm1d(64) # NEW
        self.relu5 = nn.ReLU()
        self.dropout = nn.Dropout(0.5) # Increased dropout slightly to prevent future overfitting
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.relu4(self.bn4(self.conv4(x)))
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.dropout(self.relu5(self.bn_fc1(self.fc1(x))))
        x = self.fc2(x)
        return x

model = AudioCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# NEW: Add a Learning Rate Scheduler to lower the LR when learning stalls
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

In [ ]:
import torchvision.models as models

# ==========================================
# 4. PRE-TRAINED RESNET18 ARCHITECTURE
# ==========================================
# Load a pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# ResNet expects a 3-channel (RGB) image, but our spectrograms are 1-channel (grayscale).
# We change the first convolutional layer to accept 1 channel.
old_weights = model.conv1.weight.data

model.conv1 = nn.Conv2d(
    1,
    64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

model.conv1.weight.data = old_weights.mean(dim=1, keepdim=True)

# Change the final fully connected layer to output our 10 music genres instead of 1000 ImageNet classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

# Lower learning rate because we are fine-tuning a pre-trained model
optimizer = optim.Adam(model.parameters(), lr=0.0005)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

In [ ]:
print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")



    all_preds = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())

print(Counter(all_preds))

print("Training Complete. Saving model...")
torch.save(model.state_dict(), '/content/drive/MyDrive/music_cnn_res_model.pth')

Streaming output truncated to the last 5000 lines.
TRAIN MEAN/STD: 2.7706034183502197 15.864179611206055
TRAIN SHAPE: torch.Size([1, 128, 216])
TRAIN MIN/MAX: -30.771995544433594 43.60516357421875
TRAIN MEAN/STD: 1.5701364278793335 14.525306701660156
TRAIN SHAPE: torch.Size([1, 128, 216])
TRAIN MIN/MAX: -23.94750213623047 49.155216217041016
TRAIN MEAN/STD: 11.800606727600098 11.769522666931152
TRAIN SHAPE: torch.Size([1, 128, 216])
TRAIN MIN/MAX: -61.412025451660156 45.708343505859375
TRAIN MEAN/STD: 13.41429615020752 14.323674201965332
TRAIN SHAPE: torch.Size([1, 128, 216])
TRAIN MIN/MAX: -61.24091720581055 46.700950622558594
TRAIN MEAN/STD: 9.923774719238281 14.417041778564453
TRAIN SHAPE: torch.Size([1, 128, 216])
TRAIN MIN/MAX: -33.82966613769531 43.281494140625
TRAIN MEAN/STD: 3.1963794231414795 11.781563758850098
TRAIN SHAPE: torch.Size([1, 128, 216])
TRAIN MIN/MAX: -21.223201751708984 47.2654914855957
TRAIN MEAN/STD: 14.567341804504395 9.153373718261719
TRAIN SHAPE: torch.Size([

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F

TARGET_SR = 22050
TRACK_DURATION = 5
SAMPLES_PER_TRACK = TARGET_SR * TRACK_DURATION

IDX_TO_CLASS = {
    0: "blues",
    1: "classical",
    2: "country",
    3: "disco",
    4: "hiphop",
    5: "jazz",
    6: "metal",
    7: "pop",
    8: "reggae",
    9: "rock"
}

mel_transform = T.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=2048,
    hop_length=512,
    n_mels=128
)

db_transform = T.AmplitudeToDB()


def predict_song(file_path, model, device):
    model.eval()

    waveform, sr = torchaudio.load(file_path)

    # stereo -> mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # resample
    if sr != TARGET_SR:
        resampler = T.Resample(sr, TARGET_SR)
        waveform = resampler(waveform)

    # normalize volume
    waveform = waveform / (waveform.abs().max() + 1e-9)

    total_samples = waveform.shape[1]

    # crop/pad to 30 sec
    if total_samples > SAMPLES_PER_TRACK:
        start = (total_samples - SAMPLES_PER_TRACK) // 2
        waveform = waveform[:, start:start + SAMPLES_PER_TRACK]

    else:
        pad_amount = SAMPLES_PER_TRACK - total_samples
        waveform = F.pad(waveform, (0, pad_amount))

    mel_spec = mel_transform(waveform)
    mel_spec = db_transform(mel_spec)

    mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-6)

    print(mel_spec.shape)  # should be [1,128,1292]

    mel_spec = mel_spec.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(mel_spec)

        probs = torch.softmax(outputs, dim=1)[0]

        pred_idx = torch.argmax(probs).item()

    predicted_genre = IDX_TO_CLASS[pred_idx]

    probs_dict = {
        IDX_TO_CLASS[i]: round(probs[i].item() * 100, 2)
        for i in range(len(IDX_TO_CLASS))
    }

    return predicted_genre, probs_dict

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_WEIGHTS_PATH = '/content/drive/MyDrive/music_cnn_res_model.pth'

model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model.to(device)

TEST_SONG_PATH = '/content/drive/MyDrive/The Weeknd - Save Your Tears.mp3'
song_from_dataset = '/content/drive/MyDrive/genres_original/country/country.00092.wav'

genre, probs = predict_song(TEST_SONG_PATH, model, device)

print("Predicted Genre:", genre)

for g, p in sorted(probs.items(), key=lambda x: x[1], reverse=True):
    print(f"{g}: {p}%")

torch.Size([1, 128, 216])
Predicted Genre: disco
disco: 98.24%
pop: 1.53%
hiphop: 0.11%
rock: 0.06%
country: 0.03%
reggae: 0.01%
blues: 0.0%
classical: 0.0%
jazz: 0.0%
metal: 0.0%


In [ ]:
print(CLASS_TO_IDX)

{'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [ ]:
print(IDX_TO_CLASS)

{0: 'blues', 1: 'classical', 2: 'country', 3: 'disco', 4: 'hiphop', 5: 'jazz', 6: 'metal', 7: 'pop', 8: 'reggae', 9: 'rock'}
